# UNet-64 Noise Segmentation — Training Notebook
**F-18 Chart Curve Extraction Pipeline**

Bu notebook Colab veya Kaggle üzerinde T4 GPU ile model eğitimi yapar.

**Pipeline:** Scan → Grid Removal (CV) → **UNet Noise Segmentation** → Inpaint → Clean Chart

**Model:** UNet-64, in=3 RGB, out=3 (R=arrows, G=dashed, B=text)

---
### Kullanım:
1. Bu notebook'u Colab/Kaggle'da açın
2. GPU runtime seçin (T4)
3. `synthetic_noise.py`, `model.py`, `train.py` dosyalarını yükleyin
4. Hücreleri sırayla çalıştırın

In [ ]:
# 1. GPU KONTROL
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / 1024**3
    print(f'VRAM: {mem:.1f} GB')
else:
    print('WARNING: GPU not found! Runtime > Change runtime type > T4 GPU')

## 2. Dosya Yükleme
`synthetic_noise.py`, `model.py` ve `train.py` dosyalarını yükleyin.

**Colab:** Sol panelden sürükleyip bırakın veya upload widget'ını kullanın.

**Kaggle:** Add Data > Upload > dosyaları ekleyin.

In [ ]:
# 2. DOSYA YÜKLEME
import os, sys, shutil

try:
    IN_COLAB = 'google.colab' in str(get_ipython())
except:
    IN_COLAB = False
IN_KAGGLE = os.path.exists('/kaggle/working')

REQUIRED = ['synthetic_noise.py', 'model.py', 'train.py']

if IN_COLAB:
    from google.colab import files
    print('Dosyalari yukleyin: synthetic_noise.py, model.py, train.py')
    uploaded = files.upload()
    for fname in uploaded:
        print(f'  OK: {fname}')

elif IN_KAGGLE:
    print('Kaggle ortami.')
    # Tum /kaggle/input/ agacini recursive tara
    for f in REQUIRED:
        if os.path.exists(f'/kaggle/working/{f}'):
            print(f'  OK: {f} (working)')
        else:
            found = False
            for dirpath, dirnames, filenames in os.walk('/kaggle/input'):
                if f in filenames:
                    src = os.path.join(dirpath, f)
                    shutil.copy(src, f'/kaggle/working/{f}')
                    print(f'  COPIED: {src} -> /kaggle/working/{f}')
                    found = True
                    break
            if not found:
                print(f'  MISSING: {f} — Add Data ile yukleyin')
    os.chdir('/kaggle/working')
    sys.path.insert(0, '/kaggle/working')

else:
    print('Yerel ortam.')
    for f in REQUIRED:
        status = 'OK' if os.path.exists(f) else 'MISSING'
        print(f'  {status}: {f}')

In [ ]:
# 3. IMPORT TEST
import sys
sys.path.insert(0, '.')

from model import UNet, BCEDiceLoss, model_summary
from synthetic_noise import get_torch_dataset, make_sample
from train import run_training, evaluate_model

print('\nTum moduller basariyla import edildi!')
model_summary(UNet())

## 4. Veri Onizleme
Sentetik veriden birkac ornek gorsellestirelim.

In [ ]:
# 4. VERI ONIZLEME
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for i in range(3):
    inp, mask, clean = make_sample(512, 512, seed=i*42)
    axes[i, 0].imshow(inp)
    axes[i, 0].set_title('Input (noisy)' if i == 0 else '')
    # Mask artik 2 kanalli — 3. kanal siyah
    mask_vis = np.zeros((*mask.shape[:2], 3), dtype=np.uint8)
    mask_vis[:, :, 0] = mask[:, :, 0]  # arrows -> R
    mask_vis[:, :, 1] = mask[:, :, 1]  # dashed -> G
    axes[i, 1].imshow(mask_vis)
    axes[i, 1].set_title('Mask (R=arrow G=dash)' if i == 0 else '')
    axes[i, 2].imshow(clean)
    axes[i, 2].set_title('Clean (target)' if i == 0 else '')

for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Egitim
Parametreleri asagidan degistirebilirsiniz:
- `epochs`: 30 (onerilen baslangic)
- `batch_size`: 4 (T4 16GB icin guvenli)
- `n_samples`: 6000 (epoch basina on-the-fly uretim)
- Focal loss eklemek icin `focal_weight=0.2` yapin

In [ ]:
# 5. EGITIM
history = run_training(
    epochs=30,
    batch_size=4,
    lr=1e-4,
    n_samples=6000,
    img_size=512,
    bce_weight=0.5,
    dice_weight=0.5,
    focal_weight=0.0,    # Focal eklemek icin 0.2 yapin
    use_amp=True,
    num_workers=2,
    save_dir='.',
    model_name='noise_unet.pt',
    save_every=5,
    visualize_every=5,
)

## 6. Egitim Sonuclari

In [ ]:
# 6. EGITIM GRAFIKLERI
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, len(history['epoch_loss']) + 1)

ax1.plot(epochs_range, history['epoch_loss'], 'b-', lw=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history['lr'], 'r-', lw=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('LR')
ax2.set_title('Learning Rate')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Best loss: {history["best_loss"]:.4f} (epoch {history["best_epoch"]})')
print(f'Total time: {sum(history["epoch_time"]):.0f}s ({sum(history["epoch_time"])/60:.1f}min)')

## 7. Model Degerlendirme
100 yeni sentetik ornek uzerinde IoU, Dice, Precision, Recall metrikleri.

In [ ]:
# 7. DEGERLENDIRME
results = evaluate_model('noise_unet.pt', n_samples=100)

## 8. Tahmin Ornekleri

In [ ]:
# 8. TAHMIN ORNEKLERI
import torch
import numpy as np
import matplotlib.pyplot as plt
from model import UNet
from synthetic_noise import make_sample

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet(in_channels=3, out_channels=2).to(device)
ckpt = torch.load('noise_unet.pt', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

fig, axes = plt.subplots(4, 4, figsize=(16, 16))

for row in range(4):
    inp, mask, clean = make_sample(512, 512, seed=row*17+3)
    inp_t = torch.from_numpy(inp).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    with torch.no_grad():
        pred = torch.sigmoid(model(inp_t.to(device))).squeeze(0).cpu()
    pred_bin = (pred > 0.5).float()

    axes[row, 0].imshow(inp)
    axes[row, 0].set_title('Input' if row == 0 else '')

    # 2-ch mask -> 3ch gorsel
    mask_vis = np.zeros((*mask.shape[:2], 3), dtype=np.uint8)
    mask_vis[:,:,0] = mask[:,:,0]
    mask_vis[:,:,1] = mask[:,:,1]
    axes[row, 1].imshow(mask_vis)
    axes[row, 1].set_title('GT (R=arrow G=dash)' if row == 0 else '')

    pred_vis = torch.zeros(3, pred.shape[1], pred.shape[2])
    pred_vis[0] = pred[0]
    pred_vis[1] = pred[1]
    axes[row, 2].imshow(pred_vis.permute(1, 2, 0).numpy())
    axes[row, 2].set_title('Prediction (raw)' if row == 0 else '')

    bin_vis = torch.zeros(3, pred_bin.shape[1], pred_bin.shape[2])
    bin_vis[0] = pred_bin[0]
    bin_vis[1] = pred_bin[1]
    axes[row, 3].imshow(bin_vis.permute(1, 2, 0).numpy())
    axes[row, 3].set_title('Prediction (>0.5)' if row == 0 else '')

for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 9. Model Indirme

In [ ]:
# 9. MODELI INDIR
import os

model_path = 'noise_unet.pt'
size_mb = os.path.getsize(model_path) / 1024**2
print(f'Model boyutu: {size_mb:.1f} MB')

try:
    IN_COLAB = 'google.colab' in str(get_ipython())
except:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import files
    files.download(model_path)
    print('Indirme baslatildi!')
else:
    print(f'Model kaydedildi: {os.path.abspath(model_path)}')
    print('Kaggle: Output sekmesinden indirebilirsiniz.')